# Library Imports

In [1]:
# Cell 1: Library Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Loading Data

In [2]:
# Cell 2: Data Loading
df = pd.read_csv('cleaned_diabetes.csv')
print("Dataset loaded successfully!")
print("Total rows and columns:", df.shape)

Dataset loaded successfully!
Total rows and columns: (101766, 45)


# Data Splitting

In [3]:
# Cell 3: Subsampling and Train/Test Split
# SVM time complexity is quadratic. We sample 10,000 rows to make training feasible.
df_sampled = df.sample(n=10000, random_state=42)

# Separate features (X) and target (y)
X = df_sampled.drop('readmitted', axis=1)
y = df_sampled['readmitted']

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data partitioned! Training set size: {X_train.shape[0]} rows.")

Data partitioned! Training set size: 8000 rows.


# Feature Scaling

In [4]:
# Cell 4: Feature Scaling
scaler = StandardScaler()

# Fit on training data, transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized successfully.")

Features standardized successfully.


# Base Model Training

In [5]:
# Cell 5: Train Baseline SVM
base_svm = SVC(kernel='rbf', random_state=42)
print("Training baseline SVM model...")
base_svm.fit(X_train_scaled, y_train)
print("Baseline training complete!")

Training baseline SVM model...
Baseline training complete!


# Base Model Evaluation

In [6]:
# Cell 6: Evaluate Baseline Model
y_pred_base = base_svm.predict(X_test_scaled)
base_accuracy = accuracy_score(y_test, y_pred_base)

print(f"Baseline SVM Accuracy: {base_accuracy * 100:.2f}%")
print("\nBaseline Classification Report:")
print(classification_report(y_test, y_pred_base))

Baseline SVM Accuracy: 57.10%

Baseline Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       229
           1       0.50      0.23      0.31       691
           2       0.58      0.91      0.71      1080

    accuracy                           0.57      2000
   macro avg       0.36      0.38      0.34      2000
weighted avg       0.49      0.57      0.49      2000



# Hyperparameter Tuning Setup

In [7]:
# Cell 7: GridSearchCV Setup
# Testing a small grid of parameters
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}

print("Running Grid Search to find optimal hyperparameters... (Please wait 1-3 minutes)")
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_svm = grid_search.best_estimator_
print(f"Optimal Parameters Found: {grid_search.best_params_}")

Running Grid Search to find optimal hyperparameters... (Please wait 1-3 minutes)
Optimal Parameters Found: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}


# Tuned Model Evaluation

In [8]:
# Cell 8: Evaluate Tuned Model
y_pred_tuned = best_svm.predict(X_test_scaled)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print(f"Optimized SVM Accuracy: {tuned_accuracy * 100:.2f}%")
print("\nOptimized Classification Report:")
print(classification_report(y_test, y_pred_tuned))

Optimized SVM Accuracy: 57.10%

Optimized Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       229
           1       0.50      0.23      0.31       691
           2       0.58      0.91      0.71      1080

    accuracy                           0.57      2000
   macro avg       0.36      0.38      0.34      2000
weighted avg       0.49      0.57      0.49      2000

